# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moham882/flyrank_ml_internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*My rule:
Prioritize content that appears stale and has lower-than-expected CTR for its search position, because these pages may have an opportunity for a refresh or CTR improvement.

Reason code:
STALE_LOW_CTR

Action:
Refresh / Improve CTR*

In [ ]:
staleness = (
    df.groupby(["client_hash_id", "content_hash_id"], as_index=False)
      .agg(last_seen=("report_date", "max"))
)

staleness["staleness_days"] = (
    pd.Timestamp("2026-03-31") - staleness["last_seen"]
).dt.days

print("Staleness summary:")
print(staleness["staleness_days"].describe())

staleness["staleness_bucket"] = pd.cut(
    staleness["staleness_days"],
    bins=[-1, 0, 7, 14, 21, 31],
    labels=["0 days", "1-7 days", "8-14 days", "15-21 days", "22-31 days"]
)

staleness_table = (
    staleness.groupby("staleness_bucket", observed=False)
             .agg(n=("content_hash_id", "count"))
             .reset_index()
)

print("\nStaleness bucket table:")
print(staleness_table)

Staleness summary:
count    176738.000000
mean          2.485719
std           5.919482
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max          30.000000
Name: staleness_days, dtype: float64

Staleness bucket table:
  staleness_bucket       n
0           0 days  125222
1         1-7 days   31368
2        8-14 days    8685
3       15-21 days    5418
4       22-31 days    6045


In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect()
con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

df = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM {REL}
WHERE gsc_data_available IS TRUE
""").df()

print("Rows:", len(df))
print(
    "Date range:",
    df["report_date"].min(),
    "to",
    df["report_date"].max()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 3611061
Date range: 2026-03-01 00:00:00 to 2026-03-31 00:00:00


In [ ]:
content_metrics = (
    df.groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        impressions=("gsc_impressions", "sum"),
        clicks=("gsc_clicks", "sum"),
        avg_position=("gsc_avg_position", "mean")
    )
)

content_metrics["ctr"] = (
    content_metrics["clicks"] /
    content_metrics["impressions"].replace(0, pd.NA)
)
staleness = (
    df.groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(last_seen=("report_date", "max"))
)

staleness["staleness_days"] = (
    pd.Timestamp("2026-03-31") -
    staleness["last_seen"]
).dt.days
staleness["staleness_bucket"] = pd.cut(
    staleness["staleness_days"],
    bins=[-1, 0, 7, 14, 21, 31],
    labels=[
        "0 days",
        "1-7 days",
        "8-14 days",
        "15-21 days",
        "22-31 days"
    ]
)

staleness_check = staleness.merge(
    content_metrics,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

staleness_table = (
    staleness_check
    .groupby("staleness_bucket", observed=False)
    .agg(
        n=("content_hash_id", "count"),
        median_ctr=("ctr", "median")
    )
    .reset_index()
)

print("Staleness signal check:")
print(staleness_table)

Staleness signal check:
  staleness_bucket       n  median_ctr
0           0 days  125222    0.000461
1         1-7 days   31368    0.000000
2        8-14 days    8685    0.000000
3       15-21 days    5418    0.000000
4       22-31 days    6045    0.000000


In [ ]:
ctr_position_check = content_metrics[
    content_metrics["avg_position"] > 0
].copy()
ctr_position_check["position_bucket"] = pd.cut(
    ctr_position_check["avg_position"],
    bins=[0, 3, 5, 10, 20, 50, float("inf")],
    labels=[
        "1-3",
        "4-5",
        "6-10",
        "11-20",
        "21-50",
        "51+"
    ]
)
ctr_position_table = (
    ctr_position_check
    .groupby("position_bucket", observed=False)
    .agg(
        n=("content_hash_id", "count"),
        median_ctr=("ctr", "median")
    )
    .reset_index()
)

print("CTR vs position signal check:")
print(ctr_position_table)

CTR vs position signal check:
  position_bucket      n  median_ctr
0             1-3  16144    0.000000
1             4-5  26593    0.000833
2            6-10  55395    0.000000
3           11-20  32203    0.000000
4           21-50  33288    0.000000
5             51+  11681    0.000000


## 2. Build the ranked queue (writes the CSV)

I use one simple baseline score based on the two signals checked above:

- Staleness: more days since the content was last observed receives a higher score.
- CTR vs position: content with CTR below the typical CTR for its search-position bucket receives a higher score.

The score is used only to rank content for review. It is a decision-support baseline, not a prediction of future performance.

Each row receives one reason code (`STALE_LOW_CTR`) and one action label (`Refresh / Improve CTR`). The ranked queue is written to `work/outputs/baseline_action_score.csv`.

In [ ]:
import os

os.makedirs("work/outputs", exist_ok=True)
queue = staleness_check.copy()
queue["position_bucket"] = pd.cut(
    queue["avg_position"],
    bins=[0, 3, 5, 10, 20, 50, float("inf")],
    labels=["1-3", "4-5", "6-10", "11-20", "21-50", "51+"]
)
expected_ctr = (
    queue.groupby("position_bucket", observed=False)
    .agg(
        expected_ctr=("ctr", "median")
    )
    .reset_index()
)
queue = queue.merge(
    expected_ctr,
    on="position_bucket",
    how="left"
)
queue["ctr_shortfall"] = (
    queue["expected_ctr"] - queue["ctr"]
).clip(lower=0)
queue["score"] = (
    queue["staleness_days"] +
    queue["ctr_shortfall"] * 100
)
queue["reason_code"] = "STALE_LOW_CTR"
queue["action"] = "Refresh / Improve CTR"
queue = queue.sort_values(
    ["score", "staleness_days"],
    ascending=[False, False]
).reset_index(drop=True)
output = queue[
    [
        "client_hash_id",
        "content_hash_id",
        "score",
        "reason_code",
        "action",
        "staleness_days",
        "position_bucket",
        "ctr",
        "expected_ctr"
    ]
]
output.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)
print("Ranked queue written to:")
print("work/outputs/baseline_action_score.csv")

print("\nTop 20:")
print(output.head(20))

Ranked queue written to:
work/outputs/baseline_action_score.csv

Top 20:
             client_hash_id           content_hash_id      score  \
0   client_08a6a72ff48e62c0  content_0955703807925bb8  30.083333   
1   client_08a6a72ff48e62c0  content_1a379729b59ec5f8  30.083333   
2   client_08a6a72ff48e62c0  content_1ae38248faa9e53f  30.083333   
3   client_08a6a72ff48e62c0  content_36a9b7724553fca9  30.083333   
4   client_08a6a72ff48e62c0  content_4946b174ef941ce8  30.083333   
5   client_08a6a72ff48e62c0  content_600a68ef1b184b7c  30.083333   
6   client_08a6a72ff48e62c0  content_829765455410463a  30.083333   
7   client_08a6a72ff48e62c0  content_877803e542dc5e02  30.083333   
8   client_08a6a72ff48e62c0  content_9732aa0285a08832  30.083333   
9   client_08a6a72ff48e62c0  content_ab2849de5bfcfdde  30.083333   
10  client_08a6a72ff48e62c0  content_b164ccc36230d7b4  30.083333   
11  client_08a6a72ff48e62c0  content_b950db4fc0474acd  30.083333   
12  client_08a6a72ff48e62c0  content_bd4f33

## 3. Top-20 review

*I reviewed the highest-ranked content from the baseline queue. Each review states the recommended action, the observed signals behind the ranking, and what could make the recommendation wrong.*

*These are decision-support recommendations based only on the observed March 2026 data. They should be checked before any real content change is made.*

In [ ]:
top20 = output.head(20).copy()

for i, row in top20.iterrows():
    print(
        f"{i+1}. Action: {row['action']} | "
        f"Why: {row['staleness_days']} days since last observation and "
        f"CTR {row['ctr']:.4f} vs expected {row['expected_ctr']:.4f} "
        f"for position bucket {row['position_bucket']} | "
        f"Could be wrong if: the observed gap does not represent true content staleness "
        f"or the position-based CTR comparison is not representative."
    )

1. Action: Refresh / Improve CTR | Why: 30 days since last observation and CTR 0.0000 vs expected 0.0008 for position bucket 4-5 | Could be wrong if: the observed gap does not represent true content staleness or the position-based CTR comparison is not representative.
2. Action: Refresh / Improve CTR | Why: 30 days since last observation and CTR 0.0000 vs expected 0.0008 for position bucket 4-5 | Could be wrong if: the observed gap does not represent true content staleness or the position-based CTR comparison is not representative.
3. Action: Refresh / Improve CTR | Why: 30 days since last observation and CTR 0.0000 vs expected 0.0008 for position bucket 4-5 | Could be wrong if: the observed gap does not represent true content staleness or the position-based CTR comparison is not representative.
4. Action: Refresh / Improve CTR | Why: 30 days since last observation and CTR 0.0000 vs expected 0.0008 for position bucket 4-5 | Could be wrong if: the observed gap does not represent true co

## 4. Weak picks + leakage check

*The weakest ranked items help test whether the baseline rule is producing sensible recommendations.*

*I also checked that the score uses only March 2026 observed features and does not use future-window data or a future-derived label. No client names, URLs, or private search queries are included.*

In [ ]:
# Review the weakest-ranked items

print("Bottom 10 ranked items:")
print(
    output[
        [
            "score",
            "reason_code",
            "action",
            "staleness_days",
            "position_bucket",
            "ctr",
            "expected_ctr"
        ]
    ].tail(10)
)

print("\nLeakage check:")
print("Score inputs: staleness_days and CTR shortfall")
print("Data window: March 2026 only")
print("Future-window features used: No")
print("Future-derived labels used: No")

Bottom 10 ranked items:
        score    reason_code                 action  staleness_days  \
176728    NaN  STALE_LOW_CTR  Refresh / Improve CTR               0   
176729    NaN  STALE_LOW_CTR  Refresh / Improve CTR               0   
176730    NaN  STALE_LOW_CTR  Refresh / Improve CTR               0   
176731    NaN  STALE_LOW_CTR  Refresh / Improve CTR               0   
176732    NaN  STALE_LOW_CTR  Refresh / Improve CTR               0   
176733    NaN  STALE_LOW_CTR  Refresh / Improve CTR               0   
176734    NaN  STALE_LOW_CTR  Refresh / Improve CTR               0   
176735    NaN  STALE_LOW_CTR  Refresh / Improve CTR               0   
176736    NaN  STALE_LOW_CTR  Refresh / Improve CTR               0   
176737    NaN  STALE_LOW_CTR  Refresh / Improve CTR               0   

       position_bucket  ctr  expected_ctr  
176728             NaN  0.0           NaN  
176729             NaN  0.0           NaN  
176730             NaN  0.0           NaN  
176731            

## Self-check

Before you submit, confirm each line honestly:

-✅Every section above is filled — markdown thinking AND the code that backs it

-✅The notebook runs top to bottom with no errors (Runtime → Run all)

-✅No client names, URLs, or private queries anywhere

-✅My claims use careful words: observed, measured, directional, decision-support

-✅Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.